# Minimal surfaces and membranes

A soap film spans a wire frame while trying to minimize its area. We describe the film as the graph $z=u(x,y)$. Another example is a thin membrane stretched over a fixed frame (for example a drum).

## Area and equilibrium

The area of the graph is

$$
A(u)=\int_\Omega \sqrt{1+|\nabla u|^2}\,dx.
$$

A minimizer with prescribed height on the boundary satisfies

$$
-\operatorname{div}\!\left(\frac{\nabla u}{\sqrt{1+|\nabla u|^2}}\right)=0
\quad\text{in }\Omega,
\qquad u=g\quad\text{on }\partial\Omega.
$$

For small slopes, $u\approx0$, we also can expect that $\|\nabla u\|\ll 1$ such that $\sqrt{1+|\nabla u|^2}\approx 1$. The nonlinear equation then becomes the membrane equation $-\Delta u=0$.

## An exact minimal surface

Scherk's surface

$$
u(x,y)=\log\!\left(\frac{\cos y}{\cos x}\right)
$$

solves the minimal-surface equation wherever the cosine does not vanish and the logarithm is well-defined.

In [ ]:
from netgen.occ import OCCGeometry, Rectangle
from ngsolve import (Mesh, CF, H1, GridFunction, BilinearForm,
                     Grad, dx, sqrt, Integrate, cos, log, sqrt, x, y, Integrate)
from ngsolve.webgui import Draw

half_width = 0.9
domain = Rectangle(2*half_width, 2*half_width).Face().Move((-half_width, -half_width, 0))
domain.edges.name = "wire_frame"
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=0.16))
Draw(mesh);

surface_height = log(cos(y) / cos(x))
surface_deformation = CF((0, 0, surface_height))
Draw(surface_height, mesh, "Scherk surface", deformation=surface_deformation);

## Step 1: the linear membrane problem

We first use Scherk's surface as Dirichlet data $g$ for the linearized problem. Find $u_m\in H^1(\Omega)$ with $u_m=g$ on $\partial\Omega$ such that

$$
\int_\Omega \nabla u_m\cdot\nabla v\,dx=0
\qquad\text{for all }v\in H^1_0(\Omega).
$$

This harmonic membrane will provide an initial guess for the nonlinear problem.

In [ ]:
fes = H1(mesh, order=3, dirichlet="wire_frame")
u, v = fes.TnT()

A = BilinearForm(Grad(u) * Grad(v) * dx).Assemble()
membrane_height = GridFunction(fes)
membrane_height.Set(surface_height, definedon=mesh.Boundaries("wire_frame"))

residual = membrane_height.vec.CreateVector()
residual.data = -A.mat * membrane_height.vec
inverse = A.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")
membrane_height.vec.data += inverse * residual

membrane_error = sqrt(Integrate((membrane_height - surface_height)**2, mesh)/Integrate(surface_height**2, mesh))
print(f"relative L2 error of the membrane approximation: {membrane_error:.3e}")

membrane_deformation = CF((0, 0, membrane_height))
Draw(membrane_height, mesh, "linear membrane", deformation=membrane_deformation);

## Step 2: Newton's method for the minimal surface

For

$$
R(u)(v)=\int_\Omega\frac{\nabla u\cdot\nabla v}{\sqrt{1+|\nabla u|^2}}\,dx,
$$

Newton's method computes a correction $\delta u\in H^1_0(\Omega)$ from

$$
DR(u)(\delta u,v)=-R(u)(v)
$$

and updates $u\leftarrow u+\delta u$.

In [ ]:
from ngsolve.solvers import NewtonMinimization

B = BilinearForm(fes)
B += Grad(u) * Grad(v) / sqrt(1 + Grad(u) * Grad(u)) * dx

height = GridFunction(fes)
height.vec.data = membrane_height.vec

NewtonMinimization(B, height, inverse="sparsecholesky")

error = sqrt(Integrate((height - surface_height)**2, mesh)/Integrate(surface_height**2, mesh))
print(f"Relative L2 error of the minimal-surface solution: {error:.3e}")
deformation = CF((0, 0, height))
Draw(height, mesh, "minimal surface", deformation=deformation);

## Observe

- The graph rises in one direction and falls in the other: the surface has a saddle shape (hyperbolic geometry).
- Why is Newton's error at machine precision, but the relative error of the minimal surface not? Try different polynomial orders in the `H1(order=)` space or different mesh sizes `maxh=`.


[← Mass and heat transfer](01_mass_and_heat_transfer.ipynb) · [Lecture overview](index.ipynb) · [Next: biharmonic plate →](03_biharmonic_plate.ipynb)